In [ ]:
"""
Analyse von Abhängigkeiten zwischen kategorischen Spalten in einem Polars DataFrame.

Enthält:
1. Schnelle Übersicht: welche Werte kommen zusammen vor (group_by)
2. Adjazenzmatrix (Kreuztabelle) zwischen zwei Spalten
3. Gewichtete Adjazenzmatrix (mit Häufigkeiten)
4. Gesamtgraph über alle Spalten (A-B-C) mit networkx + Visualisierung
"""

import polars as pl
import networkx as nx
import matplotlib.pyplot as plt

# -------------------------------------------------------------------
# Beispiel-Daten – ersetze das durch dein eigenes df
# -------------------------------------------------------------------
df = pl.DataFrame({
    "A": ["a1", "a1", "a2", "a2", "a3", "a1", "a2"],
    "B": ["b2", "b3", "b6", "b2", "b6", "b6", "b3"],
    "C": ["c1", "c1", "c2", "c1", "c3", "c2", "c1"],
})

In [ ]:
df

In [ ]:
def zeige_erlaubte_werte(df: pl.DataFrame, quelle: str, ziel: str) -> pl.DataFrame:
    """Für jeden Wert in `quelle`: welche Werte kommen in `ziel` vor."""
    return (
        df.select([quelle,ziel])
        .group_by(quelle)
        .all()
        .sort(quelle)        
    )

In [ ]:
print("A -> B:")
zeige_erlaubte_werte(df, "A", "B")

In [ ]:
def assoziations_matrix(df: pl.DataFrame, 
                        col1: str, 
                        col2: str, 
                        gewichtet: bool = False) -> pl.DataFrame:
    """
    Kreuztabelle zwischen col1 (Zeilen) und col2 (Spalten).
    gewichtet=False -> 1/0 ob Kombination vorkommt
    gewichtet=True  -> Anzahl der Vorkommen
    """
    if gewichtet:
        base = df.group_by([col1, col2]).len("wert")
    else:
        base = (
            df.select([col1, col2])
            .unique()
            .with_columns(pl.lit(1).alias("wert"))
        )

    return base.pivot(on=col2, index=col1, values="wert").fill_null(0).sort(col1)


In [ ]:
X= assoziations_matrix(df, "A", "B")
X

In [ ]:
okkupanz(df,"A","B")


In [ ]:
def okkupanz(df: pl.DataFrame, col1: str, col2: str):
    """Okkupanz einer ungewichteten Assoziationsmatrix."""
    base = assoziations_matrix(df, col1, col2, False).drop(col1)
    result = dict()
    # Okkupanz
    MIN = max(base.shape)
    OBS = base.sum_horizontal().sum()
    DOF = OBS-MIN
    MAX= base.width*base.height 
    REL = DOF/(MAX-MIN)
    # Return
    return {"occ_min" : MIN, 
            "occ_obs" : OBS, 
            "occ_max" : MAX, 
            "occ_dof" : DOF, # Zahl der Mehrdeutigkeiten
            "occ_rel" : REL, # relative Okkupanz
            "occ_ass" : 1-REL # Assoziation = 1 - relative Okkupanz
    }


In [ ]:
adj_AB = assoziations_matrix(df, "A", "B")
adj_AB_gewichtet = assoziations_matrix(df, "A", "B", gewichtet=True)
print("\nAdjazenzmatrix A x B (binär):")
print(adj_AB)

In [ ]:
print("\nAdjazenzmatrix A x B (mit Häufigkeiten):")
print(adj_AB_gewichtet)

In [ ]:
# Falls du ein echtes numpy-Array willst (z.B. für scikit-learn / Analyse):
adj_AB_numpy = adj_AB.drop("A").to_numpy()
print("\nAls numpy-Array:\n", adj_AB_numpy)

In [ ]:
def graph_aus_spalten(df: pl.DataFrame, 
                      spalten_paare: list[tuple[str, str]]) -> nx.Graph:
    """
    Baut einen Graphen, in dem jeder (Spalte, Wert)-Kombination ein Knoten ist.
    Kanten verbinden Werte, die in derselben Zeile gemeinsam vorkommen.
    Kantengewicht = Häufigkeit des gemeinsamen Vorkommens.
    """
    G = nx.Graph()

    for col1, col2 in spalten_paare:
        counts = df.group_by([col1, col2]).len()
        for row in counts.iter_rows(named=True):
            n1 = f"{col1}={row[col1]}"
            n2 = f"{col2}={row[col2]}"
            G.add_node(n1, layer=col1)
            G.add_node(n2, layer=col2)
            G.add_edge(n1, n2, weight=row["len"])

    return G

In [ ]:
#G = graph_aus_spalten(df, [("A", "B"), ("B", "C"), ("A", "C")])
G = graph_aus_spalten(df, [("A", "B")])

# Visualisierung: Knoten nach Spalte in Spalten/Layers angeordnet
pos = nx.multipartite_layout(G, subset_key="layer")

plt.figure(figsize=(8, 6))
weights = [G[u][v]["weight"] for u, v in G.edges()]
nx.draw(
    G, pos,
    with_labels=True,
    node_color="lightblue",
    node_size=1800,
    font_size=8,
    width=[w * 0.8 for w in weights],  # dickere Kanten = häufigere Kombination
    edge_color="gray",
)
plt.title("Abhängigkeitsgraph zwischen A, B, C")
plt.tight_layout()
plt.savefig("abhaengigkeitsgraph.png", dpi=150)
plt.show()

print("\nGraph gespeichert als abhaengigkeitsgraph.png")